# 🛰️ MISSÃO SENTINELA
### Simulado de Visão 1 · Trilha de Visão Computacional · preparação para a OIAA 2026

---

> **"Somos três analistas para olhar imagens de satélite de um estado inteiro. Quando a
> gente enxerga a pastagem nova, a mata já virou lenha faz tempo."**
>
> — Joana Batista, analista ambiental

## 📨 Pedido do núcleo de geoprocessamento

**De:** Joana Batista · Núcleo de Geoprocessamento de um órgão estadual de meio ambiente
**Para:** Equipe de Visão Computacional

> O satélite Sentinel-2 fotografa o planeta inteiro a cada cinco dias, de graça. O problema
> nunca foi falta de imagem: é falta de gente para olhar.
>
> Queremos um classificador que olhe um recorte de imagem de satélite e diga o **uso do
> solo**: **floresta**, **pastagem**, **lavoura**, **campo** (vegetação rasteira nativa) ou
> **rio**. Comparando o mesmo lugar em duas datas, dá para ver onde floresta virou pastagem.
>
> A base rotulada que existe, aberta e de boa qualidade, é o **EuroSAT**: recortes do
> Sentinel-2 **sobre a Europa**. Não é o nosso bioma. Vocês vão treinar com ela, e eu
> quero que vocês me digam, no fim, o que isso significa.
>
> Duas advertências de quem olha essas imagens todo dia. **Pastagem, campo e lavoura são o
> trio que mais confunde**: tudo é verde, tudo é baixo, e a diferença está na textura e no
> padrão. E **as classes não têm o mesmo tamanho**: há mais floresta do que campo na base,
> como há na paisagem.
>
> — Joana Batista

---

*Cenário simulado; imagens reais (Helber et al., 2019, EuroSAT, a partir de dados
Copernicus Sentinel-2), reduzidas de 64x64 para 48x48 pixels, só com as três bandas de cor
visível.*

## 🎯 A sua missão

Construir um modelo que **olha um recorte de imagem de satélite** e diz se é **floresta**,
**pastagem**, **lavoura**, **campo** ou **rio**.

Como na Missão Recomeço, cada imagem vira uma linha de uma tabela por **extração de
características**: brilho, contraste, cor média de cada canal, saturação, densidade de
bordas. O conhecimento de domínio da Joana (verde escuro e rugoso; verde claro e liso;
faixas geométricas; azul escuro contínuo) é o que diz quais medidas importam.

### 🗺️ O mapa da missão

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Ver as imagens e descobrir o que separa cada uso do solo | Só clicando |
| **2 · CONFIGURAR** | Escolher descritores, modelo e escala | Só arrastando |
| **3 · MODELAR** | Ensinar o modelo a enxergar | ✏️ Completando **9 lacunas** |
| **4 · AVALIAR** | Descobrir onde o modelo se confunde | Só clicando |

> ⚠️ **Antes de começar:** vá em `Ambiente de execução` → `Alterar o tipo de ambiente de
> execução` e confirme que está em **CPU**. Tudo aqui roda em segundos, sem placa de vídeo.
>
> Use `Arquivo` → `Salvar uma cópia no Drive` antes de começar, para não perder o trabalho.
>
> 📎 **Simulado de preparação.** Este caderno imita o formato dos desafios oficiais da Fase 1,
> com dados públicos diferentes. O gabarito do teste fica com o corretor
> (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: recebendo as imagens { display-mode: "form" }
# Os arquivos treino_fotos.npz e teste_fotos.npz ficam na mesma pasta deste caderno.
URL_TREINO_FOTOS = ""
URL_TESTE_FOTOS = ""

import pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

CLASSES = ['floresta', 'pastagem', 'lavoura', 'campo', 'rio']
EMOJI   = {'floresta': '🌳', 'pastagem': '🐄', 'lavoura': '🌾', 'campo': '🌿', 'rio': '🌊'}

def extrair_caracteristicas(fotos):
    """👀 Transforma cada FOTO em uma LINHA DE NÚMEROS que o modelo entende."""
    L = []
    for im in fotos:
        cinza = im.mean(axis=2)
        gx = np.abs(np.diff(cinza, axis=1)).mean()
        gy = np.abs(np.diff(cinza, axis=0)).mean()
        mx, mn = im.max(axis=2), im.min(axis=2)
        satur  = np.where(mx > 0, (mx - mn) / (mx + 1e-6), 0)
        L.append([im[...,0].mean(), im[...,1].mean(), im[...,2].mean(),
                  im[...,0].std(),  im[...,1].std(),  im[...,2].std(),
                  cinza.mean(), cinza.std(), np.percentile(cinza, 95),
                  np.percentile(cinza, 95) - np.percentile(cinza, 5),
                  (gx + gy) / 2, satur.mean(), satur.std(),
                  (cinza > .80).mean()])
    return pd.DataFrame(L, columns=[
        "vermelho_medio","verde_medio","azul_medio",
        "vermelho_variacao","verde_variacao","azul_variacao",
        "brilho_medio","contraste","brilho_alto",
        "amplitude_luz","bordas","saturacao_media","saturacao_variacao","area_reluzente"])

def _npz(url, nome_local):
    """Usa a cópia ao lado do caderno; senão tenta o endereço; senão pede o envio."""
    if url and not pathlib.Path(nome_local).exists():
        import urllib.request
        try:
            print(f"Baixando {nome_local}...")
            urllib.request.urlretrieve(url, nome_local)
        except Exception as erro:
            print(f"  não deu certo ({erro.__class__.__name__}).")
    if not pathlib.Path(nome_local).exists():
        from google.colab import files
        print(f"Envie o arquivo {nome_local}:")
        files.upload()
    return np.load(nome_local, allow_pickle=False)

# As fotos vêm em uint8 (0-255), como qualquer imagem; o modelo trabalha em [0,1].
_treino = _npz(URL_TREINO_FOTOS, "treino_fotos.npz")
_teste = _npz(URL_TESTE_FOTOS, "teste_fotos.npz")

fotos = _treino["fotos"].astype(np.float32) / 255.0
rotulos = _treino["rotulos"]
fotos_teste = _teste["fotos"].astype(np.float32) / 255.0
N_TESTE = len(fotos_teste)

print("Ambiente carregado.")
print(f"  Conjunto público (rotulado) ....... {len(fotos):>4} imagens, {len(CLASSES)} classes")
print(f"  Conjunto de teste (sem rótulo) .... {N_TESTE:>4} imagens  [tamanho fixo]")
print(f"  Cada imagem: {fotos.shape[1]}x{fotos.shape[2]} pixels, 3 canais de cor\n")
for k in CLASSES:
    q = (rotulos == k).sum()
    print(f"     {EMOJI[k]} {k:<14} {q:>4}   {q/len(rotulos)*100:4.1f}%")

---
# 🔍 BLOCO 1 · EXPLORAR
## *Olhem de cima, como o satélite*

> **Joana:** *"Floresta de cima é verde escuro e rugosa, cheia de sombra de copa. Pastagem é
> verde mais claro e lisa. Lavoura tem geometria: faixas, talhões, às vezes solo exposto. O
> rio é uma linha escura atravessando outra coisa."*

Cada frase dela é um descritor em potencial. "Verde escuro" é cor média e brilho; "rugosa"
é contraste e bordas; "solo exposto" é vermelho alto e saturação baixa. Verifiquem.

In [ ]:
#@title 📷 As imagens de cada classe { display-mode: "form" }
#@markdown Escolha uma classe e rode a célula (▶) para ver oito imagens dela.
classe_para_ver = "floresta"  #@param ["floresta", "pastagem", "lavoura", "campo", "rio"]

idx = np.where(rotulos == classe_para_ver)[0][:8]
fig, axes = plt.subplots(2, 4, figsize=(11, 5.6))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(fotos[i]); ax.axis("off")
fig.suptitle(f"{EMOJI[classe_para_ver]}  Amostras de {classe_para_ver.upper()}",
             fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Analisem as amostras de {classe_para_ver.upper()} sob quatro dimensões:")
print("   Luminância  - a imagem é clara ou escura, e quão uniforme é?")
print("   Cor         - qual canal domina? A cor é intensa ou apagada?")
print("   Textura     - há variação local de intensidade, ou a superfície é lisa?")
print("   Composição  - o que ocupa o centro? E a borda?")
print("\nAlternem entre as classes e registrem quais dimensões as separam melhor.")

In [ ]:
#@title 🔢 Extração de características: de imagem para vetor { display-mode: "form" }
X_caract = extrair_caracteristicas(fotos)
X_caract_teste = extrair_caracteristicas(fotos_teste)

print("Extração concluída.\n")
print(f"   Representação bruta:    {fotos.shape[1]*fotos.shape[2]*3:,} valores por imagem")
print(f"   Representação extraída: {X_caract.shape[1]} valores por imagem\n")
tabela = X_caract.copy(); tabela.insert(0, "classe", rotulos)
display(tabela.head())
print("\nA partir daqui, o problema de visão é um problema de tabela.")

In [ ]:
#@title 📊 Investigue os descritores: escolha a variável E o tipo de gráfico { display-mode: "form" }
caracteristica = "verde_medio"  #@param ["vermelho_medio", "verde_medio", "azul_medio", "vermelho_variacao", "verde_variacao", "azul_variacao", "brilho_medio", "contraste", "brilho_alto", "amplitude_luz", "bordas", "saturacao_media", "saturacao_variacao", "area_reluzente"]
tipo_de_grafico = "Boxplot"  #@param ["Boxplot", "Histograma sobreposto", "Violino", "Barras de media", "Dispersao contra outro descritor", "Galeria por faixa do descritor"]
segundo_descritor = "saturacao_media"  #@param ["vermelho_medio", "verde_medio", "azul_medio", "brilho_medio", "contraste", "bordas", "saturacao_media", "area_reluzente", "amplitude_luz"]

CORES_G = ["#2E7D32", "#C9A227", "#E6177A", "#8BC34A", "#1E88E5"]
grupos = [X_caract[rotulos == k][caracteristica] for k in CLASSES]
rot_leg = [f"{EMOJI[k]} {k}" for k in CLASSES]

fig, ax = plt.subplots(figsize=(9, 4.6))
if tipo_de_grafico == "Boxplot":
    bp = ax.boxplot(grupos, patch_artist=True, tick_labels=rot_leg,
                    medianprops=dict(color="black", linewidth=2))
    for p, cor in zip(bp["boxes"], CORES_G): p.set_facecolor(cor); p.set_alpha(.8)
    ax.set_ylabel(caracteristica)
elif tipo_de_grafico == "Histograma sobreposto":
    bins = np.histogram_bin_edges(X_caract[caracteristica], bins=28)
    for g, cor, r in zip(grupos, CORES_G, CLASSES):
        ax.hist(g, bins=bins, alpha=.5, label=r, color=cor, edgecolor="white", density=True)
    ax.legend(); ax.set_xlabel(caracteristica); ax.set_ylabel("densidade")
elif tipo_de_grafico == "Violino":
    vp = ax.violinplot(grupos, showmedians=True)
    for corpo, cor in zip(vp["bodies"], CORES_G): corpo.set_facecolor(cor); corpo.set_alpha(.75)
    ax.set_xticks(range(1, len(CLASSES) + 1)); ax.set_xticklabels(rot_leg)
    ax.set_ylabel(caracteristica)
elif tipo_de_grafico == "Barras de media":
    med = [g.mean() for g in grupos]; err = [g.std()/np.sqrt(len(g)) for g in grupos]
    ax.bar(rot_leg, med, yerr=err, capsize=7, color=CORES_G, edgecolor="#1F2A44")
    ax.set_ylabel(f"média de {caracteristica}")
elif tipo_de_grafico == "Dispersao contra outro descritor":
    for k, cor in zip(CLASSES, CORES_G):
        sub = X_caract[rotulos == k]
        ax.scatter(sub[caracteristica], sub[segundo_descritor], s=14, alpha=.55,
                   color=cor, label=k, edgecolors="none")
    ax.set_xlabel(caracteristica); ax.set_ylabel(segundo_descritor); ax.legend()
else:
    plt.close(fig)
    ordem = np.argsort(X_caract[caracteristica].values)
    fig, axes = plt.subplots(2, 5, figsize=(11.5, 5))
    for linha, grupo in ((0, ordem[:5]), (1, ordem[-5:])):
        for ax_, i in zip(axes[linha], grupo):
            ax_.imshow(fotos[i]); ax_.axis("off")
            ax_.set_title(f"{rotulos[i]}\n{X_caract[caracteristica].iloc[i]:.3f}", fontsize=8)
    fig.suptitle(f"MENORES (acima) e MAIORES (abaixo) valores de '{caracteristica}'",
                 fontweight="bold")
if tipo_de_grafico != "Galeria por faixa do descritor":
    ax.set_title(f"'{caracteristica}' por classe", fontweight="bold")
plt.tight_layout(); plt.show()
print("Um bom descritor separa as caixas. Um descritor fraco empilha todas no mesmo lugar.")
print("Anotem os pares de classes que NENHUM descritor separa: é ali que o modelo vai errar.")

> **Joana:** *"O rio vocês separam fácil. Quero ver o trio pastagem, campo e lavoura."*

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *As decisões que definem o desempenho*

| Conjunto | Tamanho | Vocês têm os rótulos? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 2.200 imagens | Sim | Treinar o modelo e validar as escolhas |
| **Teste** | **250 (fixo)** | Não | Comparação final (corrigida por `simulados/corrigir.py`) |

Os valores iniciais dos controles abaixo são **deliberadamente modestos**.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
tamanho_da_validacao = 20  #@param {type:"slider", min:10, max:40, step:5}
conjunto_de_caracteristicas = "todas"  #@param ["todas", "so cor", "so brilho e textura"]
modelo_escolhido = "SVM"  #@param ["Regressao Logistica", "Floresta Aleatoria", "SVM"]
padronizar = False  #@param {type:"boolean"}
forca_C = 1.0  #@param {type:"slider", min:0.01, max:20, step:0.01}

ORIGINAIS = ['vermelho_medio', 'verde_medio', 'azul_medio', 'vermelho_variacao', 'verde_variacao', 'azul_variacao', 'brilho_medio', 'contraste', 'brilho_alto', 'amplitude_luz', 'bordas', 'saturacao_media', 'saturacao_variacao', 'area_reluzente']
GRUPOS = {
    "todas": ORIGINAIS,
    "so cor": ["vermelho_medio", "verde_medio", "azul_medio", "saturacao_media",
               "saturacao_variacao", "vermelho_variacao", "verde_variacao", "azul_variacao"],
    "so brilho e textura": ["brilho_medio", "contraste", "brilho_alto", "amplitude_luz",
                            "bordas", "area_reluzente"],
}
if X_caract.shape[1] > len(ORIGINAIS):
    GRUPOS["todas + matiz e centro"] = list(X_caract.columns)
colunas = GRUPOS[conjunto_de_caracteristicas]

print("PARÂMETROS DEFINIDOS")
print(f"   Validação .......... {tamanho_da_validacao}% do conjunto público")
print(f"   Descritores ........ {conjunto_de_caracteristicas} ({len(colunas)} variáveis)")
print(f"   Modelo ............. {modelo_escolhido}")
print(f"   Padronizar ......... {'sim' if padronizar else 'não'}")
print(f"   Força C ............ {forca_C}")
print(f"   Teste .............. {N_TESTE} imagens  [FIXO]")

### Seleção de descritores e padronização

**Seleção.** Fornecer todos os descritores nem sempre maximiza o desempenho: variáveis
fracas adicionam ruído. Restringir o conjunto é uma forma de regularização — mas só
comparando na validação dá para saber.

**Padronização.** Os descritores têm escalas distintas (`bordas` perto de 0,05;
`brilho_medio` perto de 0,4; `area_reluzente` quase sempre 0). Modelos que operam sobre
distâncias (SVM) ou gradiente (Regressão Logística) são sensíveis a isso. Árvores não.

### 📗 Referência técnica: nome exato e parâmetros de cada classe

**Vocês vão precisar desta tabela no Bloco 3.**

> Completar só o nome da classe produz um resultado correto e mediano. Estudar esta tabela
> e **acrescentar parâmetros à chamada** produz resultados melhores. A diferença é
> intencional.

#### 🔧 Padronizador

```python
from sklearn.preprocessing import StandardScaler
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `with_mean` | `True` | Subtrai a média de cada variável |
| `with_std` | `True` | Divide pelo desvio-padrão |

#### 📈 Regressão Logística

```python
from sklearn.linear_model import LogisticRegression
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Inverso da força da regularização. **Menor regulariza mais** |
| `max_iter` | `100` | Iterações máximas. Frequentemente insuficiente |
| `class_weight` | `None` | `'balanced'` compensa desbalanceamento |
| `random_state` | `None` | Semente |

#### 🌲 Floresta Aleatória

```python
from sklearn.ensemble import RandomForestClassifier
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `n_estimators` | `100` | Número de árvores |
| `max_depth` | `None` | Profundidade máxima |
| `min_samples_leaf` | `1` | Mínimo de amostras por folha. Aumentar suaviza |
| `max_features` | `'sqrt'` | Variáveis sorteadas por corte |
| `class_weight` | `None` | `'balanced'` ou `'balanced_subsample'` |
| `n_jobs` | `None` | `-1` usa todos os núcleos |

#### 🎯 SVM

```python
from sklearn.svm import SVC
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Penalidade por erro. **Maior ajusta mais aos dados de treino** |
| `kernel` | `'rbf'` | Aceita `'linear'`, `'poly'`, `'rbf'`, `'sigmoid'` |
| `gamma` | `'scale'` | Alcance de influência de cada exemplo |
| `class_weight` | `None` | `'balanced'` compensa desbalanceamento |
| `probability` | `False` | Habilita `predict_proba`, ao custo de tempo |

> 💡 No SVM, **`C` e `gamma` interagem**. Aumentar `C` sem controlar `gamma` costuma levar a
> sobreajuste. E o SVM calcula **distâncias**: sem padronização, o descritor de maior escala
> manda sozinho.

---
# ✏️ BLOCO 3 · MODELAR
## *Ensinar o modelo a enxergar*

| Método | O que faz |
|:---|:---|
| `.fit(X, y)` | Ajusta o modelo (e o padronizador, se estiver no pipeline) ao treino |
| `.predict(X)` | Infere a classe de imagens que o modelo nunca viu |

> ⚠️ **A célula não executa enquanto houver lacunas.**

In [ ]:
# ✏️ SUA VEZ · Parte 1: separar os dados e construir o modelo
X = X_caract[colunas]
y = rotulos

# LACUNA 1 ▸ 'stratify' precisa receber o vetor cuja proporção deve ser preservada.
#            É a tabela de descritores ou o vetor de classes?
X_treino, X_valida, y_treino, y_valida = train_test_split(
    X, y, test_size=tamanho_da_validacao/100, random_state=42, stratify=____)

# LACUNAS 2 a 6 ▸ os nomes exatos das classes estão na referência do Bloco 2.
#                 Faltam também as duas palavras que continuam a cadeia condicional.
if modelo_escolhido == "Regressao Logistica":
    nucleo = ____(C=forca_C, random_state=42)

____ modelo_escolhido == "Floresta Aleatoria":
    nucleo = ____(n_estimators=250, random_state=42, n_jobs=-1)

____:
    nucleo = ____(C=forca_C, random_state=42)

# LACUNA 7 ▸ qual transformador coloca todos os descritores na mesma escala?
modelo = make_pipeline(____(), nucleo) if padronizar else nucleo

# ZONA LIVRE ▸ acrescentem parâmetros às chamadas acima. Nesta trilha, dois fatores
#   se destacam: o quanto o SVM se ajusta aos dados e se os descritores entram na mesma
#   escala. As classes não têm o mesmo tamanho: há também um parâmetro para isso.
#   Um de cada vez, comparando o F1-macro no Bloco 4.

print(f"Treino:    {len(X_treino)} imagens")
print(f"Validação: {len(X_valida)} imagens")
print(f"Modelo: {type(nucleo).__name__}{' dentro de um Pipeline com StandardScaler' if padronizar else ''}")

In [ ]:
# ✏️ SUA VEZ · Parte 2: treinar e prever
# LACUNA 8 ▸ qual método ajusta os parâmetros do modelo aos descritores de treino?
modelo.____(X_treino, y_treino)

# LACUNA 9 ▸ qual método infere as classes de imagens que o modelo nunca viu?
previsoes = modelo.____(X_valida)

print(f"Modelo ajustado. {len(previsoes)} previsões geradas para a validação.")

In [ ]:
#@title 🔎 Verificação das lacunas { display-mode: "form" }
erros = []



try:
    _, cnt_t = np.unique(y_treino, return_counts=True)
    _, cnt_a = np.unique(y, return_counts=True)
    # Estratificado: cada classe fica a no máximo ~1 imagem da composição esperada.
    if np.abs(cnt_t - cnt_a * len(y_treino) / len(y)).max() > 1.5:
        erros.append("LACUNA 1: as proporções das classes no treino diferem do conjunto "
                     "completo. O 'stratify' não recebeu o vetor correto.")
except NameError:
    erros.append("LACUNA 1: X_treino/y_treino não existem. Rodem a Parte 1.")
try:
    esperado = {"Regressao Logistica": "LogisticRegression",
                "Floresta Aleatoria": "RandomForestClassifier",
                "SVM": "SVC"}[modelo_escolhido]
    if type(nucleo).__name__ != esperado:
        erros.append(f"LACUNAS 2 a 6: para '{modelo_escolhido}' era esperada a classe "
                     f"{esperado}, mas foi construída {type(nucleo).__name__}.")
except NameError:
    erros.append("LACUNAS 2 a 6: a variável 'nucleo' não existe.")
if padronizar:
    try:
        if "standardscaler" not in dict(modelo.steps):
            erros.append("LACUNA 7: a padronização está ligada, mas o Pipeline não "
                         "contém um StandardScaler.")
    except AttributeError:
        erros.append("LACUNA 7: o Pipeline não foi construído corretamente.")
try:
    final = modelo.steps[-1][1] if hasattr(modelo, "steps") else modelo
    if not hasattr(final, "classes_"):
        erros.append("LACUNA 8: o modelo não possui parâmetros ajustados.")
except NameError: pass
try:
    if len(previsoes) != len(X_valida):
        erros.append(f"LACUNA 9: {len(previsoes)} previsões para {len(X_valida)} casos.")
except NameError:
    erros.append("LACUNA 9: a variável 'previsoes' não existe.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
else:
    print("VERIFICAÇÃO APROVADA\n")
    pad = {k: v for k, v in nucleo.get_params().items()
           if k in ("class_weight", "min_samples_leaf", "max_features", "gamma", "kernel",
                    "max_depth", "max_iter")
           and v not in (None, 1, "sqrt", "scale", "rbf", 100)}
    if pad:
        print(f"   Parâmetros adicionais detectados na ZONA LIVRE: {pad}")
    else:
        print("   Nenhum parâmetro adicional declarado: o modelo usa os valores padrão.")

---
# 📈 BLOCO 4 · AVALIAR
## *Onde a máquina se confunde?*

> **Joana:** *"Para a fiscalização, dois erros me importam. Floresta chamada de pastagem
> manda uma equipe de campo atrás de um desmatamento que não existe. Pastagem chamada de
> floresta esconde um desmatamento que existe. O primeiro custa diesel. O segundo custa
> mata."*

In [ ]:
#@title 🏆 O resultado { display-mode: "form" }
acuracia = accuracy_score(y_valida, previsoes)
f1 = f1_score(y_valida, previsoes, average="macro")

print("="*60)
print("           DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*60)
print(f"  Acurácia (métrica de apoio) .......... {acuracia*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) ......... {f1*100:5.1f}%")
print("="*60)
# Faixas calibradas nestes descritores.
if   f1 >= .87: print("  Desempenho alto. Olhem agora quais pares ainda se confundem.")
elif f1 >= .82: print("  Desempenho sólido, com margem de otimização no Bloco 2.")
elif f1 >= .75: print("  Desempenho intermediário. Revisem padronização e descritores.")
else:           print("  Desempenho baixo. A representação atual não separa as classes.")

cm = confusion_matrix(y_valida, previsoes, labels=CLASSES)
fig, ax = plt.subplots(figsize=(6.6, 5.6))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="viridis", colorbar=False)
ax.set_title("Onde a máquina acertou e onde se confundiu", fontweight="bold")
ax.set_xlabel("O que a máquina DISSE"); ax.set_ylabel("O que REALMENTE era")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

confusoes = sorted([(CLASSES[i], CLASSES[j], cm[i, j]) for i in range(len(CLASSES))
                    for j in range(len(CLASSES)) if i != j and cm[i, j] > 0],
                   key=lambda t: -t[2])
print("\n🔎 MAIORES CONFUSÕES DO MODELO:")
for real, disse, q in confusoes[:3]:
    print(f"   • {q} imagens de {EMOJI[real]} {real.upper()} foram chamadas de "
          f"{EMOJI[disse]} {disse.upper()}")
print()
print(classification_report(y_valida, previsoes, labels=CLASSES, digits=3))
if not padronizar and modelo_escolhido in ("Regressao Logistica", "SVM"):
    print("[!] O modelo escolhido é sensível à escala e a padronização está DESLIGADA.")
    print("    Ligá-la no Bloco 2 costuma ser o ganho isolado mais relevante nesta trilha.")

In [ ]:
#@title 🕵️ A galeria dos erros { display-mode: "form" }
idx_valida = X_valida.index.to_numpy()
y_valida_arr = np.asarray(y_valida)
errou = np.where(previsoes != y_valida_arr)[0]
if len(errou) == 0:
    print("🎯 Nenhum erro na validação.")
else:
    k = min(8, len(errou))
    fig, axes = plt.subplots(1, k, figsize=(2.1*k, 2.9))
    axes = np.atleast_1d(axes)
    for ax, e in zip(axes, errou[:k]):
        ax.imshow(fotos[idx_valida[e]]); ax.axis("off")
        ax.set_title(f"era {y_valida_arr[e]}\ndisse {previsoes[e]}", fontsize=8,
                     color="#B00020", fontweight="bold")
    fig.suptitle("Imagens que enganaram a máquina", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
    print(f"Classificações incorretas: {len(errou)} de {len(y_valida)} imagens.\n")
    print("   Uma pessoa acertaria essas imagens, nesta resolução de 48x48?")
    print("   Os erros têm algo em comum (escuras, cor apagada, muito fundo)?")
    print("   Que medida nova capturaria a diferença que os descritores atuais perdem?")

---
# 🤔 PARADA OBRIGATÓRIA: Europa não é Cerrado

> **Joana:** *"Duas perguntas, e nenhuma é sobre o código."*

### Questões para a equipe deliberar

1. **Deslocamento de domínio.** O modelo aprendeu como é uma floresta **europeia** vista de
   cima: temperada, de copa fechada, com a luz e as estações de lá. O Cerrado tem
   formações savânicas, campo sujo, cerradão, veredas, estação seca que deixa tudo marrom.
   Que classes vocês esperam que piorem no Cerrado, e por quê? Como testariam isso antes
   de usar o modelo aqui, e o que seria preciso para corrigir?

2. **Da imagem à multa.** Um alerta de desmatamento pode virar auto de infração, embargo
   de propriedade, bloqueio de crédito rural. Com a taxa de erro que vocês mediram, o
   alerta do modelo pode ser prova? Ou só um motivo para mandar alguém olhar?

3. **Quem é mais vigiado.** Se o modelo erra mais em certas paisagens (propriedades
   pequenas, com mosaico de usos, por exemplo), quem recebe mais alarmes falsos? Isso
   é só um problema técnico?

In [ ]:
#@title 📝 O relatório da sua equipe { display-mode: "form" }
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Quais descritores vocês usaram e por quê?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. Quais classes o modelo mais confundiu? Por que isso acontece?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. Como vocês testariam se o modelo funciona no Cerrado antes de usá-lo lá?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. O modelo pode ser usado para multar alguém? Em que condições?**
resposta_4 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido,
             "caracteristicas": conjunto_de_caracteristicas, "padronizar": padronizar,
             "C": forca_C, "validacao_pct": tamanho_da_validacao,
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "just_caracteristicas": resposta_1, "just_confusoes": resposta_2,
             "just_dominio": resposta_3, "just_uso": resposta_4}
faltando = [k for k in ["equipe", "just_caracteristicas", "just_confusoes",
                        "just_dominio", "just_uso"]
            if not str(relatorio[k]).strip()]
if faltando:
    print("⚠️ Ainda faltam campos:", ", ".join(faltando))
else:
    print("Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
import json, pathlib

PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}

palpites = modelo.predict(X_caract_teste[colunas])
submissao = pd.DataFrame({"id_foto": np.arange(len(palpites)),
                          "classe_prevista": palpites})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_sentinela.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_sentinela.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_sentinela.csv   ({len(submissao)} linhas)")
print("  relatorio_sentinela.json")
print("="*60)
print("\n  Distribuição prevista no teste:")
for k, v in pd.Series(palpites).value_counts().items():
    print(f"     {EMOJI[k]} {k:<14} {v:>4}")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py sentinela {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA MISSÃO SENTINELA

> **Joana:** *"Rodamos o modelo em cem recortes do Cerrado que a equipe já tinha
> classificado à mão. O resultado caiu, como vocês avisaram que cairia, e caiu exatamente
> nas classes que vocês apontaram. Agora sabemos o que rotular primeiro para retreinar.
> Um modelo que avisa onde vai falhar vale mais do que um que promete acertar tudo."*

---

### Competências praticadas neste simulado

- Extração de características de imagens de sensoriamento remoto
- Conhecimento de domínio orientando a escolha de descritores
- Padronização e sensibilidade de SVM/Regressão Logística à escala
- Leitura da matriz de confusão com custos diferentes por par de classes
- **Deslocamento de domínio**: treinar num bioma e aplicar em outro
- Alerta automatizado × evidência: o limite do uso punitivo de um modelo